# 03 — Interpretabilidade (T13 e T14)

Feature importance e SHAP para os modelos de `02_avaliacao.ipynb`. Split e modelos são reproduzidos com os mesmos parâmetros (`random_state=42`). A classe positiva é **M (maligno)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

df = pd.read_csv("../data/data.csv").drop(columns=['id', 'Unnamed: 32'])
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42)

def make_model(classifier):
    pre = ColumnTransformer(transformers=[('num', Pipeline(steps=[('scaler', StandardScaler())]), list(X.columns))])
    return Pipeline(steps=[('preprocessor', pre), ('classifier', classifier)])

modelo_lr = make_model(LogisticRegression(random_state=42)).fit(X_train, y_train)
modelo_rf = make_model(RandomForestClassifier(random_state=42)).fit(X_train, y_train)

## T13 — Feature importance

### Random Forest: importância por impureza\n\nCalculada no treino; pode favorecer features correlacionadas e contínuas, por isso é comparada com a importância por permutação.

In [ ]:
imp_rf = pd.Series(modelo_rf.named_steps['classifier'].feature_importances_, index=X.columns).sort_values()
imp_rf.tail(10).plot.barh(figsize=(7, 4), title="Random Forest — importância por impureza (top 10)")
plt.tight_layout()
plt.savefig("../reports/figures/importancia_rf_impureza.png", dpi=150)
plt.show()
imp_rf.sort_values(ascending=False).head(10).round(3)

### Importância por permutação (conjunto de validação)\n\nQueda do recall de M ao embaralhar cada feature. Usa a validação para não tocar o teste.

In [ ]:
from sklearn.metrics import make_scorer, recall_score
scorer = make_scorer(recall_score, pos_label='M')
perm = permutation_importance(modelo_rf, X_val, y_val, scoring=scorer, n_repeats=30, random_state=42)
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values()
imp_perm.tail(10).plot.barh(figsize=(7, 4), title="Random Forest — importância por permutação (recall de M)")
plt.tight_layout()
plt.savefig("../reports/figures/importancia_rf_permutacao.png", dpi=150)
plt.show()
imp_perm.sort_values(ascending=False).head(10).round(4)

### Regressão Logística: coeficientes (features padronizadas)

In [ ]:
coef = pd.Series(modelo_lr.named_steps['classifier'].coef_[0], index=X.columns)
assert list(modelo_lr.named_steps['classifier'].classes_) == ['B', 'M']  # coef positivo => aumenta P(maligno)
top = coef.reindex(coef.abs().sort_values().index).tail(10)
top.plot.barh(figsize=(7, 4), title="Regressão Logística — coeficientes (top 10 por |valor|)")
plt.tight_layout()
plt.savefig("../reports/figures/coeficientes_lr.png", dpi=150)
plt.show()
top.sort_values(ascending=False).round(3)

## T14 — SHAP\n\nExplica a contribuição de cada feature para a probabilidade/log-odds de **maligno**, nas amostras do teste (nenhuma decisão de modelo é tomada com base nisso).

In [ ]:
def dados_transformados(modelo, Xd):
    pre = modelo.named_steps['preprocessor']
    return pd.DataFrame(pre.transform(Xd), columns=X.columns, index=Xd.index)

# Regressão Logística (linear)
Xtr_lr = dados_transformados(modelo_lr, X_train)
Xte_lr = dados_transformados(modelo_lr, X_test)
expl_lr = shap.LinearExplainer(modelo_lr.named_steps['classifier'], Xtr_lr)
sv_lr = expl_lr(Xte_lr)

shap.summary_plot(sv_lr.values, Xte_lr, show=False)
plt.title("SHAP — Regressão Logística (teste)")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_lr.png", dpi=150)
plt.show()

In [ ]:
# Random Forest (árvores)
Xte_rf = dados_transformados(modelo_rf, X_test)
expl_rf = shap.TreeExplainer(modelo_rf.named_steps['classifier'])
sv_rf = expl_rf(Xte_rf)
idx_m = list(modelo_rf.named_steps['classifier'].classes_).index('M')
vals_rf = sv_rf.values[:, :, idx_m] if sv_rf.values.ndim == 3 else sv_rf.values

shap.summary_plot(vals_rf, Xte_rf, show=False)
plt.title("SHAP — Random Forest (teste, classe M)")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_rf.png", dpi=150)
plt.show()

In [ ]:
# Importância global média |SHAP| dos dois modelos
comp = pd.DataFrame({
    'LR': np.abs(sv_lr.values).mean(axis=0),
    'RF': np.abs(vals_rf).mean(axis=0),
}, index=X.columns)
comp['LR'] /= comp['LR'].sum(); comp['RF'] /= comp['RF'].sum()
comp.sort_values('RF', ascending=False).head(10).round(3)

## Conclusões T13 e T14

- **Random Forest:** as features mais importantes, tanto por impureza quanto por SHAP, são as medidas `worst` de tamanho e concavidade: `area_worst`, `concave points_worst`, `perimeter_worst` e `radius_worst`. No SHAP, valores altos delas aumentam a probabilidade de maligno e valores baixos a reduzem, o que é coerente com o conhecimento clínico (tumores maiores e com contorno mais irregular são mais suspeitos).
- **Multicolinearidade:** `radius`, `perimeter` e `area` são quase a mesma informação. Por isso a importância por permutação é baixa e dispersa (a maior queda de recall é ~0,05, em `concave points_worst`): ao embaralhar uma delas, as correlatas compensam. Importâncias individuais dessas features não devem ser lidas isoladamente, e sim como um grupo "tamanho do núcleo".
- **Regressão Logística:** os coeficientes são instáveis pela mesma razão. Aparecem `radius_se` no topo e `compactness_se` com sinal negativo, que não têm leitura clínica direta. O SHAP da regressão é mais distribuído entre as features do que o do Random Forest. Os coeficientes servem para o modelo, mas não devem ser interpretados como efeito individual de cada variável.
- **Uso no contexto clínico:** SHAP permite mostrar ao médico quais características levaram a cada predição, mas explica o modelo e não a doença: é correlação aprendida nesta base, sem garantia de causalidade. A decisão continua sendo do médico.
